In [1]:
import pandas as pd
import csv

# Configuração robusta para lidar com aspas e símbolos em poemas
params = {
    'sep': '\t', 
    'names': ['text', 'label'], 
    'header': None, 
    'on_bad_lines': 'skip',
    'quoting': csv.QUOTE_NONE  # Impede que aspas no meio do verso quebrem a leitura
}

# Carga dos datasets
train_df = pd.read_csv('data/train.tsv', **params)
test_df = pd.read_csv('data/test.tsv', **params)
dev_df = pd.read_csv('data/dev.tsv', **params)

# Verificação inicial
print(f"Dataset de Treino: {len(train_df)} versos")
train_df.head()

Dataset de Treino: 892 versos


,text,label
0,with pale blue berries. in these peaceful shad...,1
1,"it flows so long as falls the rain,",0
2,"and that is why, the lonesome day,",-1
3,"when i peruse the conquered fame of heroes, an...",2
4,of inward strife for truth and liberty.,2


## 2. Etapas do Pipeline de PLN e Pré-processamento

Nesta etapa, realizamos dois experimentos de pré-processamento para comparar o impacto na limpeza dos versos:

1. **Experimento 1 (Simples):** Padronização do texto em minúsculas e remoção de caracteres especiais/pontuação.
2. **Experimento 2 (Avançado):** Além da padronização, aplicamos a remoção de *Stopwords* (palavras irrelevantes) e o *Stemming* (redução das palavras ao seu radical).

O objetivo é transformar a linguagem poética, rica em conectivos, em uma representação mais densa e focada em palavras-chave de sentimento.

In [2]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download silencioso de recursos necessários
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def limpar_texto(text, modo_avancado=False):
    # Padronização básica (Experimento 1)
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    
    # Limpeza profunda (Experimento 2)
    if modo_avancado:
        tokens = text.split()
        text = " ".join([stemmer.stem(w) for w in tokens if w not in stop_words])
    return text

# Aplicação dos experimentos no dataset de treino
train_df['simples'] = train_df['text'].apply(lambda x: limpar_texto(x, modo_avancado=False))
train_df['avancado'] = train_df['text'].apply(lambda x: limpar_texto(x, modo_avancado=True))

# Visualização do comparativo (essencial para a explicação do pipeline)
train_df[['text', 'simples', 'avancado']].head()

,text,simples,avancado
0,with pale blue berries. in these peaceful shad...,with pale blue berries in these peaceful shades,pale blue berri peac shade
1,"it flows so long as falls the rain,",it flows so long as falls the rain,flow long fall rain
2,"and that is why, the lonesome day,",and that is why the lonesome day,lonesom day
3,"when i peruse the conquered fame of heroes, an...",when i peruse the conquered fame of heroes and...,perus conquer fame hero victori mighti gener e...
4,of inward strife for truth and liberty.,of inward strife for truth and liberty,inward strife truth liberti


## 3. Representação Textual e Classificação Tradicional (Naive Bayes)

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# 1. Representação Textual (TF-IDF)
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(train_df['avancado'])

# 2. Modelo de Classificação 1: Naive Bayes
modelo_nb = MultinomialNB()
modelo_nb.fit(X_tfidf, train_df['label'])

# 3. Validação simples
previsoes = modelo_nb.predict(X_tfidf)
acc = accuracy_score(train_df['label'], previsoes)

print(f"Representação TF-IDF: {X_tfidf.shape[1]} palavras únicas mapeadas.")
print(f"Acurácia do Naive Bayes: {acc:.2%}")

Representação TF-IDF: 1890 palavras únicas mapeadas.
Acurácia do Naive Bayes: 69.51%


## 4. Análise de Similaridade Textual

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculando a similaridade entre o primeiro verso e todos os outros
similaridade = cosine_similarity(X_tfidf[0], X_tfidf)
index_similar = similaridade.argsort()[0][-2] # Pega o segundo mais similar (o primeiro é ele mesmo)

print(f"Verso Original: {train_df['text'].iloc[0]}")
print(f"Verso Similar:  {train_df['text'].iloc[index_similar]}")
print(f"Grau de Similaridade: {similaridade[0][index_similar]:.2f}")

Verso Original: with pale blue berries. in these peaceful shades--
Verso Similar:  on that shaded day,
Grau de Similaridade: 0.34


4. Geração de Texto: Comparativo RNN vs LSTM

In [6]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SimpleRNN
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Preparação dos dados para as Redes Neurais
tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_df['avancado'])
total_palavras = len(tokenizer.word_index) + 1

input_sequences = []
for line in train_df['avancado']:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

max_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_len, padding='pre'))
X_gen, y_gen = input_sequences[:,:-1], tf.keras.utils.to_categorical(input_sequences[:,-1], num_classes=total_palavras)

# 2. Definição das duas arquiteturas para comparação
def criar_modelo(tipo='RNN'):
    model = Sequential([
        Embedding(total_palavras, 50, input_length=max_len-1),
        SimpleRNN(100) if tipo == 'RNN' else LSTM(100),
        Dense(total_palavras, activation='softmax')
    ])
    model.compile(loss='categorical_crossentropy', optimizer='adam')
    return model

# 3. Treino
print("Treinando RNN...")
rnn_model = criar_modelo('RNN')
rnn_model.fit(X_gen, y_gen, epochs=15, verbose=0)

print("Treinando LSTM...")
lstm_model = criar_modelo('LSTM')
lstm_model.fit(X_gen, y_gen, epochs=15, verbose=0)

# 4. Função para gerar texto
def gerar_verso(modelo, texto_semente, proximas_palavras=5):
    for _ in range(proximas_palavras):
        tokens = tokenizer.texts_to_sequences([texto_semente])[0]
        tokens = pad_sequences([tokens], maxlen=max_len-1, padding='pre')
        previsao = np.argmax(modelo.predict(tokens, verbose=0), axis=-1)
        
        saida = ""
        for palavra, index in tokenizer.word_index.items():
            if index == previsao:
                saida = palavra
                break
        texto_semente += " " + saida
    return texto_semente

# Exibição dos resultados para comparação
semente = "love"
print(f"\nSemente: {semente}")
print(f"Gerado pela RNN:  {gerar_verso(rnn_model, semente)}")
print(f"Gerado pela LSTM: {gerar_verso(lstm_model, semente)}")

Treinando RNN...


c:\Users\J\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Treinando LSTM...

Semente: love
Gerado pela RNN:  love round fli seem know good
Gerado pela LSTM: love like poet chime crown easi
